# 03 · Análisis exploratorio sobre Bronze

| | |
|---|---|
| **Fase CRISP-DM** | 2 — Comprensión de los datos (EDA) |
| **Objetivo** | Describir la calidad y el comportamiento de los datos para definir las reglas de limpieza de la capa Silver. |
| **Entradas** | `data/bronze/voltajes_EL_A.parquet`, `data/bronze/voltajes_EL_B.parquet` |
| **Salidas** | Figuras en `reports/eda/` y lista de reglas de calidad (sección 8) |

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap

from voltcast import config, io

config.ensure_dirs()
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
ELS = list(config.SHEETS)
COL_EL = {"A": "tab:blue", "B": "tab:orange"}


def savefig(fig, name):
    path = config.EDA_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")


bronze = {el: io.read_bronze(el) for el in ELS}
schema = {el: io.detect_schema(df) for el, df in bronze.items()}
CELLS = schema["A"].cell_cols
for el, df in bronze.items():
    print(f"EL {el}: {df.shape[0]} filas, {schema[el].n_cells} celdas")

In [ ]:
# Vistas de trabajo para el EDA (en memoria; Bronze no se modifica).
# Se indexa por fecha calendario y se conserva el último registro de cada día.
def vista_diaria(df):
    d = df.assign(fecha=df["Date"].dt.normalize()).sort_values("Date")
    return d.drop_duplicates("fecha", keep="last").set_index("fecha")

dia = {el: vista_diaria(df) for el, df in bronze.items()}
V = {el: d[CELLS] for el, d in dia.items()}  # voltaje de las celdas
KA = {el: d["[kA]"] for el, d in dia.items()}
VT = {el: d["[Volts]"] for el, d in dia.items()}

## 1. Fechas: rango, frecuencia, hora de registro y duplicados

In [ ]:
filas = []
for el, df in bronze.items():
    d = df["Date"]
    dias = d.dt.normalize()
    calendario = pd.date_range(dias.min(), dias.max(), freq="D")
    h = d.dt.hour
    filas.append({
        "EL": el,
        "desde": dias.min().date(), "hasta": dias.max().date(),
        "registros": len(d), "días calendario": len(calendario),
        "días sin registro": len(calendario.difference(dias)),
        "fechas duplicadas": int(dias.duplicated().sum()),
        "% a las 00:00": round(100 * (h == 0).mean(), 1),
        "% 07:00–11:59": round(100 * h.between(7, 11).mean(), 1),
        "% otra hora": round(100 * (~((h == 0) | h.between(7, 11))).mean(), 1),
    })
pd.DataFrame(filas).set_index("EL")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for el, df in bronze.items():
    h = df["Date"].dt.hour + df["Date"].dt.minute / 60
    axes[0].hist(h, bins=48, range=(0, 24), alpha=0.6, label=f"EL {el}", color=COL_EL[el])
    axes[1].scatter(df["Date"], h, s=2, alpha=0.5, label=f"EL {el}", color=COL_EL[el])
axes[0].set(title="Hora del registro", xlabel="hora del día", ylabel="registros")
axes[1].set(title="Hora del registro a lo largo del tiempo", ylabel="hora del día")
axes[0].legend(); axes[1].legend(markerscale=4)
fig.tight_layout(); savefig(fig, "01_hora_registro")

for el, df in bronze.items():
    d = df["Date"]
    print(f"EL {el}: primer registro a las 00:00 el {d[d.dt.hour == 0].min().date()}")
    dup = d[d.dt.normalize().duplicated(keep=False)]
    print(f"   registros con fecha duplicada: {dup.dt.strftime('%Y-%m-%d %H:%M').tolist()}")

**Lectura.** Hasta septiembre de 2020 el registro se toma por la mañana (07:00–11:00); desde el 27-09-2020 se toma a las 00:00. Cada fila representa un día, así que en Silver la marca de tiempo se reduce a **fecha calendario**. Hay pocos días duplicados (varios registros el mismo día), que se resuelven quedándose con el último, y algunos días sin registro, que quedan como huecos.

## 2. Valores faltantes por fecha y por celda

In [ ]:
falt = pd.DataFrame({
    el: {
        "celdas×días NaN": int(V[el].isna().sum().sum()),
        "% NaN celdas": round(100 * V[el].isna().mean().mean(), 2),
        "días sin ninguna celda": int(V[el].isna().all(axis=1).sum()),
        "días con alguna celda NaN": int(V[el].isna().any(axis=1).sum()),
        "[kA] NaN": int(KA[el].isna().sum()),
        "[Volts] NaN": int(VT[el].isna().sum()),
    } for el in ELS
}).T
falt

In [ ]:
# Celdas con más días sin dato
top = pd.DataFrame({el: V[el].isna().sum() for el in ELS})
top["total"] = top.sum(axis=1)
top.sort_values("total", ascending=False).head(10)

### Mapa de estado celdas × tiempo

Cada punto combina faltantes y valores anómalos: `ok`, `NaN` (sin dato), `paro` (kA < `KA_SHUTDOWN`), `V < 0,5` con kA normal (celda puenteada o fuera de servicio) y `V > 5` (atípico). Se agrega por semana tomando el estado más frecuente.

In [ ]:
ESTADOS = ["ok", "NaN", "paro", "V<0.5", "V>5"]
CMAP = ListedColormap(["#e8f0e8", "#bdbdbd", "#6a51a3", "#d7301f", "#fdae61"])


def matriz_estado(el):
    v, ka = V[el], KA[el]
    paro = (ka < config.KA_SHUTDOWN).to_numpy()[:, None]
    est = np.zeros(v.shape, dtype=int)
    est[v.isna().to_numpy()] = 1
    est[(v < config.V_OUT_OF_SERVICE).to_numpy() & ~paro] = 3
    est[(v > config.V_MAX_PLAUSIBLE).to_numpy() & ~paro] = 4
    est[np.broadcast_to(paro, v.shape)] = 2
    # Calendario continuo: los días sin registro cuentan como NaN
    full = pd.date_range(v.index.min(), v.index.max(), freq="D")
    return pd.DataFrame(est, index=v.index, columns=v.columns).reindex(full, fill_value=1)


estado = {el: matriz_estado(el) for el in ELS}
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for ax, el in zip(axes, ELS):
    sem = estado[el].resample("W").agg(lambda s: s.mode().iloc[0])
    ax.imshow(sem.T.to_numpy(), aspect="auto", cmap=CMAP, vmin=-0.5, vmax=4.5, interpolation="nearest",
              extent=[mdates.date2num(sem.index[0]), mdates.date2num(sem.index[-1]), len(CELLS) + 0.5, 0.5])
    ax.xaxis_date(); ax.set(ylabel="celda", title=f"EL {el} — estado por celda (semanal)")
handles = [plt.Rectangle((0, 0), 1, 1, color=CMAP(i)) for i in range(len(ESTADOS))]
fig.legend(handles, ESTADOS, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(); savefig(fig, "02_mapa_estado_celdas")

pd.DataFrame({el: pd.Series(estado[el].to_numpy().ravel()).map(dict(enumerate(ESTADOS))).value_counts(normalize=True).mul(100).round(2)
              for el in ELS})

## 3. Días de paro (kA bajo) y su efecto sobre los voltajes

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, el in zip(axes, ELS):
    ka = KA[el]
    ax.plot(ka.index, ka, lw=0.8, color=COL_EL[el])
    p = ka[ka < config.KA_SHUTDOWN]
    ax.scatter(p.index, p, color="crimson", s=10, zorder=3, label=f"paro (kA < {config.KA_SHUTDOWN}): {len(p)} días")
    ax.axhline(config.KA_SHUTDOWN, ls="--", color="grey", lw=0.8)
    ax.set(ylabel="kA", title=f"EL {el} — corriente"); ax.legend(loc="lower left")
fig.tight_layout(); savefig(fig, "03_corriente_paros")

In [ ]:
# Efecto del paro: voltaje medio de las celdas en servicio frente a la corriente
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, el in zip(axes, ELS):
    vm = V[el].where(V[el] >= config.V_OUT_OF_SERVICE).mean(axis=1)
    paro = KA[el] < config.KA_SHUTDOWN
    ax.scatter(KA[el][~paro], vm[~paro], s=4, alpha=0.4, label="operación normal")
    ax.scatter(KA[el][paro], V[el][paro].mean(axis=1), s=12, color="crimson", label="paro (todas las celdas)")
    ax.set(xlabel="kA", ylabel="V medio de las celdas", title=f"EL {el}"); ax.legend()
fig.tight_layout(); savefig(fig, "04_efecto_paro")

for el in ELS:
    paro = KA[el] < config.KA_SHUTDOWN
    vp = V[el][paro]
    print(f"EL {el}: {paro.sum()} días de paro | V de celdas en paro: mediana {np.nanmedian(vp):.2f} V, "
          f"min {np.nanmin(vp):.2f}, max {np.nanmax(vp):.2f} | celdas×días negativos: {(vp < 0).sum().sum()}")

**Lectura.** En los días de paro la corriente cae a ~0 kA y los voltajes de las celdas dejan de reflejar su estado: bajan a valores cercanos a 0, algunos negativos o por encima de 5 V. Esos días no sirven como observación del voltaje de operación y se marcan como `paro`.

## 4. Celdas fuera de servicio o puenteadas

In [ ]:
def tramos(mask, min_dias=7, tolerancia=5):
    """Tramos de días donde ``mask`` es True, uniendo interrupciones de hasta ``tolerancia`` días."""
    idx = mask.index[mask.to_numpy()]
    if len(idx) == 0:
        return []
    cortes = np.where(np.diff(idx.values).astype("timedelta64[D]").astype(int) > tolerancia)[0]
    ini = np.r_[0, cortes + 1]; fin = np.r_[cortes, len(idx) - 1]
    return [(idx[a].date(), idx[b].date(), (idx[b] - idx[a]).days + 1) for a, b in zip(ini, fin)
            if (idx[b] - idx[a]).days + 1 >= min_dias]


filas = []
for el in ELS:
    ka_ok = KA[el] >= config.KA_SHUTDOWN
    full = pd.date_range(V[el].index.min(), V[el].index.max(), freq="D")
    v = V[el].reindex(full)
    ka_ok = ka_ok.reindex(full, fill_value=False)
    for c in CELLS:
        bajo = (v[c] < config.V_OUT_OF_SERVICE) & ka_ok  # puenteada: V < 0,5 V (o negativo) con kA normal
        sin_dato = v[c].isna() & v.notna().sum(axis=1).gt(len(CELLS) // 2)  # la celda falta pero el resto sí tiene dato
        for tipo, m in (("V<0.5 con kA normal", bajo), ("sin dato (resto con dato)", sin_dato)):
            for a, b, n in tramos(m, min_dias=7):
                filas.append({"EL": el, "celda": int(c), "tipo": tipo, "desde": a, "hasta": b, "días": n})
fuera = pd.DataFrame(filas).sort_values(["EL", "celda", "desde"])
fuera.groupby(["EL", "celda", "tipo"]).agg(desde=("desde", "min"), hasta=("hasta", "max"),
                                           dias=("días", "sum"), tramos=("días", "size"))

In [ ]:
# Evolución de las celdas puenteadas conocidas (61, 62, 123, 124 de ambos EL y 46 del EL B)
casos = {"A": ["61", "62", "123", "124"], "B": ["46", "61", "62", "123", "124"]}
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, el in zip(axes, ELS):
    for c in casos[el]:
        ax.plot(V[el].index, V[el][c], lw=0.8, label=f"celda {c}")
    ax.plot(V[el].index, V[el].median(axis=1), color="k", lw=1, ls="--", label="mediana de celdas")
    ax.axhline(config.V_OUT_OF_SERVICE, color="crimson", lw=0.8, ls=":")
    ax.set(ylim=(-1.5, 4), ylabel="V", title=f"EL {el} — celdas fuera de servicio / puenteadas"); ax.legend(ncol=6, fontsize=8)
fig.tight_layout(); savefig(fig, "05_celdas_puenteadas")

In [ ]:
# Celdas en servicio por día (V >= 0,5 con kA normal)
fig, ax = plt.subplots(figsize=(14, 3.5))
for el in ELS:
    ok = KA[el] >= config.KA_SHUTDOWN
    n = (V[el] >= config.V_OUT_OF_SERVICE).sum(axis=1)[ok]
    ax.plot(n.index, n, lw=1, label=f"EL {el}", color=COL_EL[el])
ax.set(ylabel="celdas con V ≥ 0,5", title="Celdas en servicio (días sin paro)"); ax.legend()
fig.tight_layout(); savefig(fig, "06_celdas_en_servicio")

**Lectura.** Las celdas **61, 62, 123 y 124** de ambos electrolizadores están puenteadas durante un período largo, que empieza **antes de lo esperado**:
- desde el inicio de la serie (2020-01) hasta ~may-2022 registran V < 0,5 V o negativo con kA normal;
- desde ~may-2022 hasta ~oct-2024 casi no tienen dato (NaN) mientras el resto de las celdas sí lo tiene, con intervalos cortos de V < 0,5;
- vuelven a operar a partir de ~oct-2024 (en el EL A, 123 y 124 con algún tramo más hasta ~abr-2025).

Además, la **celda 46 del EL B** está puenteada entre jul-2021 y ene-2022, y hay tramos cortos en otras celdas del EL B: 18, 60, 122 y 132 en ene-2020 (≤ 4 semanas) y 80 entre ago y oct-2022 (sin dato y luego V < 0,5).

La especificación preveía el tramo may-2022 → oct-2024; los datos muestran que ese tramo es solo la parte **sin dato**. La regla de Silver debe cubrir las dos formas: V < 0,5 con kA normal **y** ausencia de dato prolongada.

## 5. Tendencia de envejecimiento, estacionalidad y dispersión entre celdas

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for el in ELS:
    ok = KA[el] >= config.KA_SHUTDOWN
    vm = V[el].where(V[el] >= config.V_OUT_OF_SERVICE)[ok].mean(axis=1)
    axes[0].plot(VT[el][ok].index, VT[el][ok].rolling(30, min_periods=10).median(), color=COL_EL[el], label=f"EL {el}")
    axes[1].plot(vm.index, vm, lw=0.5, alpha=0.4, color=COL_EL[el])
    axes[1].plot(vm.index, vm.rolling(30, min_periods=10).median(), color=COL_EL[el], label=f"EL {el}")
axes[0].set(ylabel="V", title="[Volts] total (mediana móvil de 30 días, sin paros)")
axes[1].set(ylabel="V", title="Voltaje medio de las celdas en servicio (diario y mediana móvil de 30 días)")
axes[0].legend(); axes[1].legend()
fig.tight_layout(); savefig(fig, "07_tendencia_envejecimiento")

anual = pd.concat({el: pd.DataFrame({
    "[Volts]": VT[el][KA[el] >= config.KA_SHUTDOWN],
    "V medio celdas": V[el].where(V[el] >= config.V_OUT_OF_SERVICE)[KA[el] >= config.KA_SHUTDOWN].mean(axis=1),
    "kA": KA[el][KA[el] >= config.KA_SHUTDOWN],
}).groupby(lambda d: d.year).mean().round(3) for el in ELS}, axis=1)
anual

In [ ]:
# Estacionalidad: desvío del voltaje medio respecto de su tendencia (mediana móvil centrada de 365 días), por mes
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, el in zip(axes, ELS):
    ok = KA[el] >= config.KA_SHUTDOWN
    vm = V[el].where(V[el] >= config.V_OUT_OF_SERVICE)[ok].mean(axis=1).asfreq("D")
    desv = (vm - vm.rolling(365, center=True, min_periods=180).median()).dropna()
    sns.boxplot(x=desv.index.month, y=desv.values, ax=ax, color=COL_EL[el], fliersize=1)
    ax.set(xlabel="mes", ylabel="V (desvío de la tendencia)", title=f"EL {el} — estacionalidad mensual")
fig.tight_layout(); savefig(fig, "08_estacionalidad")

In [ ]:
# Dispersión entre celdas: percentiles diarios del voltaje de las celdas en servicio
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, el in zip(axes, ELS):
    ok = KA[el] >= config.KA_SHUTDOWN
    v = V[el].where(V[el] >= config.V_OUT_OF_SERVICE)[ok]
    q = v.quantile([0.05, 0.25, 0.5, 0.75, 0.95], axis=1).T
    ax.fill_between(q.index, q[0.05], q[0.95], alpha=0.2, color=COL_EL[el], label="p5–p95")
    ax.fill_between(q.index, q[0.25], q[0.75], alpha=0.4, color=COL_EL[el], label="p25–p75")
    ax.plot(q.index, q[0.5], color="k", lw=0.8, label="mediana")
    ax.set(ylim=(2.6, 3.8), ylabel="V", title=f"EL {el} — dispersión entre celdas"); ax.legend(loc="upper left", ncol=3)
fig.tight_layout(); savefig(fig, "09_dispersion_celdas")

In [ ]:
# Nivel medio de cada celda según su posición en el stack (último año con datos)
fig, ax = plt.subplots(figsize=(14, 3.8))
for el in ELS:
    ok = KA[el] >= config.KA_SHUTDOWN
    v = V[el].where(V[el] >= config.V_OUT_OF_SERVICE)[ok]
    nivel = v.loc[v.index.max() - pd.Timedelta(days=365):].mean()
    ax.plot(range(1, len(CELLS) + 1), nivel.values, marker=".", lw=0.8, label=f"EL {el}", color=COL_EL[el])
ax.set(xlabel="posición de la celda", ylabel="V medio (último año)", title="Nivel de voltaje por posición en el stack"); ax.legend()
fig.tight_layout(); savefig(fig, "10_nivel_por_posicion")

**Lectura.**
- **Envejecimiento.** El voltaje medio **no crece de forma monótona**: tiene forma de **diente de sierra**, con tramos de subida lenta (degradación) interrumpidos por **caídas bruscas** (intervenciones sobre un lote de celdas, §5.1). En conjunto baja entre 2021 y 2023 y vuelve a subir desde 2024. El modelo debe seguir el nivel reciente (de ahí el **ancla** de los últimos `ANCHOR_DAYS` días) y proyectar la pendiente.
- **Estacionalidad.** El desvío mensual respecto de la tendencia es pequeño frente a la tendencia y a la dispersión entre celdas; no justifica covariables de calendario en la PoC.
- **Dispersión.** Las celdas difieren en nivel (unas décimas de voltio) de forma persistente, lo que justifica el embedding de identidad de celda y el objetivo relativo al ancla.

### 5.1 Saltos de nivel por celda (intervenciones)

La banda p5–p95 sube lentamente y cae de golpe: grupos de celdas bajan varias décimas de voltio de un día para otro. Se detectan como caídas de más de 0,1 V entre la mediana de los 7 días previos y la de los 7 días siguientes (sin paros ni celdas puenteadas).

In [ ]:
def saltos(el, umbral=0.10, ventana=7):
    """Primer día de cada caída de nivel > ``umbral`` V por celda (mediana 7 días antes vs. 7 días después)."""
    full = pd.date_range(V[el].index.min(), V[el].index.max(), freq="D")
    ok = KA[el].reindex(full) >= config.KA_SHUTDOWN
    v = V[el].reindex(full).where(ok, axis=0)
    v = v.where(v >= config.V_OUT_OF_SERVICE)
    antes = v.rolling(ventana, min_periods=4).median()
    despues = v[::-1].rolling(ventana, min_periods=4).median()[::-1].shift(-1)
    delta = despues - antes
    cae = delta < -umbral
    inicio = cae & ~cae.shift(1, fill_value=False)
    return inicio, delta


eventos = []
fig, ax = plt.subplots(figsize=(14, 3.5))
for el in ELS:
    inicio, delta = saltos(el)
    n = inicio.sum(axis=1)
    ax.bar(n.index, n.values, width=4, color=COL_EL[el], alpha=0.7, label=f"EL {el}")
    for t, k in n[n >= 3].items():
        eventos.append({"EL": el, "fecha": t.date(), "celdas": int(k),
                        "caída mediana (V)": round(float(delta.loc[t][inicio.loc[t]].median()), 3)})
    print(f"EL {el}: {int(inicio.sum().sum())} saltos > 0,1 V en {int((inicio.sum() > 0).sum())} de {len(CELLS)} celdas")
ax.set(ylabel="celdas con salto", title="Celdas que inician una caída de nivel > 0,1 V, por día"); ax.legend()
fig.tight_layout(); savefig(fig, "12_saltos_de_nivel")
pd.DataFrame(eventos).pivot_table(index="fecha", columns="EL", values=["celdas", "caída mediana (V)"])

**Lectura.** Casi todas las celdas tienen al menos un salto de nivel, y los saltos se concentran en unas 12–14 fechas por electrolizador (varias comunes a ambos), con caídas típicas de 0,2–0,7 V. Hay una intervención cada ~6 meses, **el mismo orden que el horizonte de pronóstico**. Algunas coinciden con el arranque tras un paro.

Implicaciones para el modelado:
- Un pronóstico a 180 días que cruce una intervención tendrá un error del orden de la caída; el modelo no puede anticiparla con la información disponible (no hay registro de mantenimientos).
- En la evaluación (notebook 07) conviene reportar las métricas **también separando** los orígenes cuyo horizonte cruza un salto, para no confundir error del modelo con intervenciones.
- El ancla de los últimos `ANCHOR_DAYS` días permite que el modelo se reajuste rápido después de cada salto.

## 6. Relación entre `[Volts]` y las celdas, y correlación con kA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
tabla = []
for el in ELS:
    ok = KA[el] >= config.KA_SHUTDOWN
    v = V[el].where(V[el] >= config.V_OUT_OF_SERVICE)[ok]
    ratio = VT[el][ok] / v.mean(axis=1)
    n = v.notna().sum(axis=1)
    axes[0].plot(ratio.index, ratio, lw=0.8, label=f"EL {el}", color=COL_EL[el])
    exceso = VT[el][ok] - v.sum(axis=1)  # voltaje no explicado por las celdas (barras, conexiones)
    tabla.append(pd.DataFrame({"EL": el, "celdas en servicio": n, "[Volts]/media": ratio, "[Volts] − Σceldas": exceso}))
axes[0].set(ylabel="[Volts] / media de celdas", title="Relación [Volts] / voltaje medio de las celdas"); axes[0].legend()

# Correlación de cada celda con kA (días sin paro)
for el in ELS:
    ok = KA[el] >= config.KA_SHUTDOWN
    corr = V[el][ok].where(V[el][ok] >= config.V_OUT_OF_SERVICE).corrwith(KA[el][ok])
    axes[1].hist(corr.dropna(), bins=30, alpha=0.6, label=f"EL {el} (mediana {corr.median():.2f})", color=COL_EL[el])
axes[1].set(xlabel="correlación de Pearson", ylabel="celdas", title="Correlación de cada celda con kA (sin paros)"); axes[1].legend()
fig.tight_layout(); savefig(fig, "11_volts_total_y_kA")

tabla = pd.concat(tabla)
tabla[tabla["celdas en servicio"] > 0].groupby(["EL", "celdas en servicio"]).agg(dias=("[Volts]/media", "size"),
                                              ratio_mediana=("[Volts]/media", "median"),
                                              exceso_mediano_V=("[Volts] − Σceldas", "median")).round(2)

In [ ]:
# La columna 'Average' del Excel frente a la media calculada de las celdas
for el, df in bronze.items():
    avg = df["Average"]
    calc = df[CELLS].mean(axis=1)
    print(f"EL {el}: 'Average' con dato en {avg.notna().sum()} de {len(df)} filas | "
          f"|Average − media| mediana {np.nanmedian((avg - calc).abs()):.4f} V, máx {np.nanmax((avg - calc).abs()):.4f} V")

**Lectura.**
- `[Volts]` / media de las celdas en servicio **no es constante (≈ 183)**: vale ≈ 180 con 177 celdas en servicio y ≈ 186 con las 181. Es decir, `[Volts]` ≈ Σ celdas en servicio + ~9–15 V de barras y conexiones (con variaciones). Por eso la covariable del electrolizador se define como `V_total / n_celdas_en_servicio`, que se ajusta automáticamente a los puenteos.
- La correlación de las celdas con kA en operación normal es **débil y de signo variable** (negativa en la mayoría de las celdas del EL A y ligeramente positiva en el EL B), porque la corriente casi no varía fuera de los paros. Aun así, `kA` se mantiene como covariable para que el modelo detecte los cambios de régimen.
- La columna `Average` del Excel solo tiene dato en ~60 % de las filas y es redundante con la media de las celdas: se descarta en Silver.

## 7. Valores atípicos

In [ ]:
for el in ELS:
    ka_ok = KA[el] >= config.KA_SHUTDOWN
    v = V[el]
    neg = (v < 0) & ka_ok.to_numpy()[:, None]
    alto = (v > config.V_MAX_PLAUSIBLE) & ka_ok.to_numpy()[:, None]
    print(f"EL {el}: V < 0 → {int((v < 0).sum().sum())} celdas×día en total, "
          f"{int(neg.sum().sum())} con kA normal (celdas: {sorted(neg.columns[neg.any()].astype(int).tolist())})")
    print(f"      V > {config.V_MAX_PLAUSIBLE} → {int((v > config.V_MAX_PLAUSIBLE).sum().sum())} en total, "
          f"{int(alto.sum().sum())} con kA normal")

**Lectura.** Fuera de las celdas puenteadas, los valores negativos y los mayores a 5 V aparecen **solo en días de paro**. La regla de `atipico` (V > 5 V o V < 0 aislados) queda como salvaguarda para datos futuros.

## 8. Conclusión: reglas de calidad para Silver

| # | Regla | Justificación (EDA) |
|---|---|---|
| R1 | Renombrar columnas a `A_V_total`, `A_kA`, `A_c001..A_c181` (ídem `B_`) y **descartar `Average`**. | `Average` está incompleta y es redundante (§6). |
| R2 | `Date` → **fecha calendario**; ordenar; en fechas duplicadas quedarse con el **último registro del día**; reindexar a frecuencia **diaria continua**. | La hora de registro cambió en sep-2020; hay fechas duplicadas y días sin registro (§1). |
| R3 | Interpolar linealmente los huecos de hasta `MAX_INTERP_DAYS` = 3 días (`interpolado`); los más largos quedan `NaN` (`hueco`). | Días sin registro y NaN aislados (§1, §2). |
| R4 | `paro`: `kA < KA_SHUTDOWN` (10 kA). Esos días no son observaciones válidas del voltaje. | Voltajes ~0, negativos o > 5 V con kA ≈ 0 (§3, §7). |
| R5 | `fuera_servicio`: tramos (≥ 7 días, tolerando interrupciones cortas) con **V < 0,5 V (incluye negativos) y kA normal**, **o sin dato durante un período largo** mientras el resto de las celdas sí lo tiene. | Celdas 61, 62, 123 y 124 de ambos EL (2020 → ~oct-2024, primero con V ≈ 0 y luego sin dato) y celda 46 del EL B (jul-2021 → ene-2022) (§4). |
| R6 | `atipico`: V > 5 V o V < 0 aislados con kA normal → `NaN`. | Hoy solo ocurre en paros; se mantiene como salvaguarda (§7). |
| R7 | Covariable del electrolizador: `V_total / n_celdas_en_servicio`, no una constante. | `[Volts]` / media varía entre ≈ 180 y ≈ 186 según las celdas en servicio (§6). |
| R8 | Split temporal 80/20 decidido en Silver y reutilizado por todas las capas siguientes. | Evita fugas de información; la tendencia no es estacionaria (§5). |
| R9 | *(Propuesta, no incluida en la especificación v3)* Registrar los **saltos de nivel** por celda (fecha, caída) como tabla de eventos en Silver. No se corrigen los valores; se usan para analizar el error en la evaluación. | ~220–240 saltos de 0,1–0,7 V por electrolizador, cada ~6 meses (§5.1). |

Estas reglas se implementan en `src/voltcast/cleaning.py` y se aplican en el notebook **04 — Preparación Silver**.